# Probability 02 — Random Variables, Expectation, and Variance

## Objective

Represent a discrete probability distribution, compute its moments, simulate Bernoulli/binomial outcomes, and compare actions by expected cost.

**Prerequisites:** Probability 01: events, conditional probability, and a valid sample space.

**Working rule:** Complete the starter cells yourself. Verification cells are deliberately placed after each task and will fail until that task is solved. Keep the assertions; catch only failures that the task explicitly asks you to provoke. Use NumPy plus Python's standard library only. After finishing, restart the kernel and run all cells in order.

**Why this matters:** ML decisions can be framed as expected losses under uncertainty, not just as labels or raw scores.


## Fixed distribution

Let `Y` be the number of suspicious signals recorded for a transaction. This toy variable has values 0 through 3. Its probabilities are synthetic and sum to one.

In [ ]:
import numpy as np

values = np.array([0., 1., 2., 3.])
pmf = np.array([.50, .30, .15, .05])
assert values.shape == pmf.shape == (4,)


## Task 1 — Validate a probability mass function

### Goal

Make explicit what counts as a discrete distribution.

### Required work

1. Implement `validate_pmf(values, probabilities)` returning numerical arrays.
2. Reject negative probabilities, unequal lengths, empty arrays, and totals not near one.
3. Explain why values need not themselves sum to one.

### Expected output

The supplied distribution is accepted; four invalid cases are rejected.

### Acceptance criteria

- [ ] Probabilities are nonnegative and sum to one within tolerance.
- [ ] Invalid inputs raise descriptive `ValueError`s.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def validate_pmf(values, probabilities):
    # TODO: convert and validate; return both 1D arrays.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
x, p = validate_pmf(values, pmf)
assert x.shape == p.shape == (4,)
for x_bad, p_bad in ((values, [.5, .3, .2]), (values, [.5, .3, .1, .05]),
                     (values, [.5, .6, -.2, .1]), ([], [])):
    try:
        validate_pmf(x_bad, p_bad)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid PMF was accepted')


## Task 2 — Expectation and population variance

### Goal

Connect weighted averages and spread to a random variable.

### Required work

1. Implement `pmf_moments(values, probabilities)` using loops or elementary NumPy arithmetic, not `np.average` inside the function.
2. Calculate `E[Y]`, `E[Y²]`, and `Var(Y)` by hand for the fixed distribution.
3. Compare the function with `np.average` and explain variance units.

### Expected output

Mean `0.75`; population variance `0.7875`.

### Acceptance criteria

- [ ] Function calls `validate_pmf`.
- [ ] Variance is nonnegative within numerical tolerance.
- [ ] Hand and NumPy checks agree.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def pmf_moments(values, probabilities):
    # TODO: return (mean, variance), using the PMF.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
mean_y, var_y = pmf_moments(values, pmf)
assert np.isclose(mean_y, .75)
assert np.isclose(var_y, .7875)
assert np.isclose(mean_y, np.average(values, weights=pmf))
assert np.isclose(var_y, np.average((values - mean_y)**2, weights=pmf))


## Task 3 — Bernoulli and binomial reasoning

### Goal

Connect binary outcomes to counts over repeated trials.

### Required work

1. Implement `bernoulli_moments(probability)` and `binomial_moments(trials, probability)`.
2. For five independent alert opportunities with probability `0.2` each, predict expected alert count and variance.
3. Reject probabilities outside `[0,1]` and negative trial counts.

### Expected output

For `(n=5, p=.2)`, expected count `1.0` and variance `.8`.

### Acceptance criteria

- [ ] Boundary probabilities 0 and 1 behave correctly.
- [ ] The independence assumption is stated.
- [ ] Invalid parameters raise `ValueError`.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def bernoulli_moments(probability):
    # TODO: return (mean, variance).
    raise NotImplementedError

def binomial_moments(trials, probability):
    # TODO: return (mean, variance).
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert np.allclose(bernoulli_moments(.2), (.2, .16))
assert np.allclose(binomial_moments(5, .2), (1., .8))
assert np.allclose(binomial_moments(5, 0), (0, 0))
for args in ((5, -.1), (5, 1.1), (-1, .2)):
    try:
        binomial_moments(*args)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid binomial inputs were accepted')


## Task 4 — Seeded simulation and sampling variation

### Goal

Compare theoretical moments with simulated counts.

### Required work

1. Use `np.random.default_rng(42)` to simulate at least 20,000 draws from `Binomial(5, .2)`.
2. Compute empirical mean and population variance; compare them with Task 3.
3. Explain why an individual sample does not have to equal a theoretical expectation.

### Expected output

Empirical mean near 1 and variance near `.8`.

### Acceptance criteria

- [ ] Simulation is reproducible.
- [ ] Empirical statistics are within stated tolerances.
- [ ] Population and sample variance conventions are distinguished.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: create draws, empirical_mean, empirical_variance.
draws = empirical_mean = empirical_variance = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert draws is not None and len(draws) >= 20_000
assert empirical_mean is not None and abs(empirical_mean - 1.) < .04
assert empirical_variance is not None and abs(empirical_variance - .8) < .05
assert np.isclose(empirical_variance, np.var(draws, ddof=0))


## Task 5 — Expected cost of a fraud review

### Goal

Choose an action by expected cost rather than raw event probability alone.

### Required work

1. Implement `expected_costs(fraud_probability, fraud_loss, review_cost, prevention_fraction)` returning costs of no review and review.
2. Assume review costs 3 units and prevents 80% of an otherwise 100-unit fraud loss; compare actions at fraud probabilities `.08` and `.01`.
3. Explain the role of cost assumptions in a real ML threshold.

### Expected output

At `.08`, no-review cost `8` and review cost `4.6`; at `.01`, no-review cost `1` and review cost `3.2`.

### Acceptance criteria

- [ ] Costs are nonnegative for valid inputs.
- [ ] The preferred action flips between the two cases.
- [ ] Invalid probabilities/fractions are rejected.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def expected_costs(fraud_probability, fraud_loss, review_cost, prevention_fraction):
    # TODO: validate inputs and return (no_review_cost, review_cost_total).
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert np.allclose(expected_costs(.08, 100, 3, .8), (8., 4.6))
assert np.allclose(expected_costs(.01, 100, 3, .8), (1., 3.2))
for p in (-.1, 1.1):
    try:
        expected_costs(p, 100, 3, .8)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid fraud probability was accepted')


## Task 6 — Debug variance and units

### Goal

Avoid confusing a PMF variance with a finite-sample estimate.

### Required work

1. Using the simulated draws, compute variance with `ddof=0` and `ddof=1`; report their difference.
2. Explain which one Task 2 calculates and why a finite sample has another convention.
3. Scale `Y` by 10 and verify how mean and variance change; state the new units.

### Expected output

Mean scales by 10; variance scales by 100.

### Acceptance criteria

- [ ] Both variance conventions are shown.
- [ ] Scaling assertions pass.
- [ ] The explanation includes squared units for variance.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: assign sample_variance, scaled_mean, scaled_variance.
sample_variance = scaled_mean = scaled_variance = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert sample_variance is not None and np.isclose(sample_variance, np.var(draws, ddof=1))
assert scaled_mean is not None and np.isclose(scaled_mean, 10 * mean_y)
assert scaled_variance is not None and np.isclose(scaled_variance, 100 * var_y)


## Completion check

- [ ] All task cells are implemented and all provided assertions pass.
- [ ] Expected exceptions are caught inside the relevant test cells.
- [ ] **Restart Kernel and Run All** succeeds in notebook order.
- [ ] Each task has a short written explanation in your own words.
- [ ] No real private data, credentials, or large generated files were added.

**Conceptual exit questions**

1. Why must a PMF sum to one?
2. What does expectation mean when no observation equals the mean?
3. What independence assumption underlies binomial variance?
4. Why can a low fraud probability still justify review?

**Final deliverable:** A notebook with validated PMFs, theoretical and simulated moments, an expected-cost decision, and explanations of variance conventions.
